In [25]:
import numpy as np
import pandas as pd

# Побудова брудного набору для Варіанта 1 (Київ, базова ціна 850, кількість 3)
dirty_data = [
    {"місто": "Київ", "ціна": 850, "кількість": 3},
    {"місто": " Київ ", "ціна": "850 грн", "кількість": np.nan},  # пропуск кількості, рядок замість числа, зайві пробіли в місті
    {"місто": "КИЇВ", "ціна": 850, "кількість": 2},                  # написання великими літерами
    {"місто": "Kyiv", "ціна": "850 грн", "кількість": np.nan},  # пропуск кількості, латиниця, рядок замість числа
    {"місто": "Київ", "ціна": 850, "кількість": 3},                  # повний дублікат першого рядка
    {"місто": "Київ", "ціна": 850, "кількість": 4},
    {"місто": "Київ", "ціна": 850, "кількість": 5},
    {"місто": "Київ", "ціна": 8500, "кількість": 3}                  # викид (ціна перевищує базову в 10 разів)
]

df = pd.DataFrame(dirty_data)
print("Брудний DataFrame:")
display(df)

Брудний DataFrame:


,місто,ціна,кількість
0,Київ,850,3.0
1,Київ,850 грн,NaN
2,КИЇВ,850,2.0
3,Kyiv,850 грн,NaN
4,Київ,850,3.0
5,Київ,850,4.0
6,Київ,850,5.0
7,Київ,8500,3.0


In [26]:
# 1. Визначення кількості пропусків у кожному стовпці
print("Кількість пропусків:\n", df.isna().sum())

# 2. Заповнення пропусків у стовпці "кількість" медіаною
median_qty = df["кількість"].median()
df["кількість"] = df["кількість"].fillna(median_qty)
print(f"Медіана кількості, якою заповнено пропуски: {median_qty}")

Кількість пропусків:
 місто        0
ціна         0
кількість    2
dtype: int64
Медіана кількості, якою заповнено пропуски: 3.0


In [27]:
# 1. Пошук дублікатів без subset (повний збіг рядка) та з subset
print("Кількість дублікатів (без subset):", df.duplicated().sum())
print("Кількість дублікатів (за subset=['місто', 'ціна', 'кількість']):",
      df.duplicated(subset=["місто", "ціна", "кількість"]).sum())

# 2. Видалення дублікатів
df = df.drop_duplicates().reset_index(drop=True)
print("Кількість рядків після видалення дублікатів:", len(df))

Кількість дублікатів (без subset): 1
Кількість дублікатів (за subset=['місто', 'ціна', 'кількість']): 1
Кількість рядків після видалення дублікатів: 7


In [28]:
# 1. Переведення стовпця "ціна" у числовий тип (float)
df["ціна"] = df["ціна"].astype(str).str.replace(" грн", "").astype(float)

# 2. Узгодження назв міст (приведення до нижнього регістру, видалення пробілів, заміна латиниці)
df["місто"] = df["місто"].str.strip().str.lower().replace({"kyiv": "київ"})

print("Унікальні значення стовпця 'місто' після узгодження:", df["місто"].unique())

Унікальні значення стовпця 'місто' після узгодження: ['київ']


In [29]:
# 1. Обчислення меж IQR для стовпця "ціна"
q1, q3 = df["ціна"].quantile([0.25, 0.75])
iqr = q3 - q1
lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

print(f"Q1: {q1} | Q3: {q3} | IQR: {iqr}")
print(f"Нижня межа: {lower_bound} | Верхня межа: {upper_bound}")

# Виведення потенційних викидів
outliers = df[(df["ціна"] < lower_bound) | (df["ціна"] > upper_bound)]
print("Знайдені викиди:\n", outliers)

Q1: 850.0 | Q3: 850.0 | IQR: 0.0
Нижня межа: 850.0 | Верхня межа: 850.0
Знайдені викиди:
   місто    ціна  кількість
6  київ  8500.0        3.0


### Відповіді на контрольні запитання:

1. **Чому заповнення пропуску середнім (а не медіаною) може згодом спотворити стандартне відхилення:**
   - Середнє арифметичне дуже чутливе до викидів і може зміщуватися в бік аномальних значень, тоді як медіана стійкіша. Крім того, штучне заповнення великої кількості пропусків однаковим значенням (середнім чи медіаною) штучно сплощує дисперсію та занижує реальне стандартне відхилення вибірки, оскільки додає «середніх» точок, які не мають природної варіативності.

2. **Різниця між duplicated() без subset і з subset:**
   - `duplicated()` без аргументів перевіряє повний збіг усіх стовпців у рядку. Використання параметра `subset` дозволяє обмежити перевірку лише конкретними ключовими полями (наприклад, ID замовлення або сукупністю основних параметрів), ігноруюрчи можливі розбіжності в другорядних службових стовпцях, що запобігає хибно-негативним результатам.

3. **Чому автоматичне видалення всіх значень поза межами IQR не завжди правильне рішення:**
   - Критерій IQR є лише статистичним інструментом для пошуку статистичних аномалій. Не всі значення за межами меж IQR є помилками чи брудними даними — це можуть бути реальні екстремальні явища (наприклад, рекордні продажі, унікальні дорогі замовлення), видалення яких призведе до втрати важливої цінної інформації про хвіст розподілу.